# Reproduce and check *When Does Cross-Market Transfer Help?* in Google Colab

Run the cells **from top to bottom** (Runtime → Run all works too). No GPU is needed.

* **Part A (about 10 minutes, no raw data needed):** installs the pinned environment, runs the repository's
  self-check and unit tests, rebuilds every table and figure from the shipped results and confirms that the
  tables, supplementary tables, result files and figure data (the CSV files behind each figure) are identical to
  the published ones. The figure PDFs/PNGs themselves are redrawn but not byte-compared.
* **Part B (optional, about 15 minutes, needs the raw XMarket zip, e.g. in your Google Drive):** rebuilds the
  preprocessed data from the raw files and re-runs real experiments on a few stores, then compares every record
  with the published results. It is skipped when `DATA_ZIP` is left empty (the default).

Success looks like `PART A PASSED` and, if `DATA_ZIP` is set, `PART B PASSED`.

## 1. Settings — edit these two lines only
* `CODE_SOURCE`: the GitHub address of the repository **or** the path of the code zip in your Google Drive.
* `DATA_ZIP`: path of the raw XMarket zip, e.g. in your Google Drive (only needed for Part B; leave empty to skip Part B).
  Google Drive is connected only if one of the two paths starts with `/content/drive`.

In [ ]:
CODE_SOURCE = "https://github.com/basmakdd/xmarket-transfer"   # or e.g. "/content/drive/MyDrive/xmarket-transfer-v1.0-files.zip"
DATA_ZIP    = ""   # raw XMarket ratings zip (Part B only; empty = skip Part B), e.g. "/content/drive/MyDrive/xmarket_categories.zip"

import os
WORK = os.environ.get("XM_NB_WORK", "/content")                    # working folder (Colab: /content)
CODE_SOURCE = os.environ.get("XM_NB_CODE", CODE_SOURCE)
DATA_ZIP = os.environ.get("XM_NB_DATA", DATA_ZIP) or ""
print("code:", CODE_SOURCE); print("data:", DATA_ZIP or "(none: Part B will be skipped)")

## 2. Connect Google Drive (only if the code zip or the data zip is in Drive)

In [ ]:
try:
    from google.colab import drive
    if CODE_SOURCE.startswith("/content/drive") or (DATA_ZIP and DATA_ZIP.startswith("/content/drive")):
        drive.mount("/content/drive")
    else:
        print("No Google Drive path given: Drive is not connected.")
except ImportError:
    print("Not running in Colab: skipping Google Drive.")

## 3. Get the code

In [ ]:
import os, glob, shutil, subprocess, zipfile
REPO = os.path.join(WORK, "xmarket-transfer")
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if CODE_SOURCE.startswith("http"):
    subprocess.run(["git", "clone", "--depth", "1", CODE_SOURCE, REPO], check=True)
else:
    tmp = os.path.join(WORK, "_code_unzip")
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(CODE_SOURCE).extractall(tmp)
    # the repository root is the folder holding run.py (zip with or without a top folder; macOS __MACOSX/ ignored)
    hits = sorted((h for h in glob.glob(os.path.join(tmp, "**", "run.py"), recursive=True) if "__MACOSX" not in h),
                  key=lambda h: (h.count(os.sep), h))
    assert hits, "run.py not found in the code zip"
    shutil.move(os.path.dirname(hits[0]), REPO)
os.chdir(REPO)
print(sorted(os.listdir(REPO)))

## 4. Install the pinned environment
The repository pins exact package versions (Python 3.10–3.13). To avoid clashing with Colab's own packages,
this cell creates a separate environment with `uv` (takes 1–2 minutes).

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
ENV = os.path.join(WORK, "xm-env")
subprocess.run(["uv", "venv", "--python", "3.12", ENV], check=True)
subprocess.run(["uv", "pip", "install", "--python", os.path.join(ENV, "bin", "python"), "-q", "-r", "requirements.txt"], check=True)
PY = os.path.join(ENV, "bin", "python")
print(subprocess.run([PY, "-c", "import numpy, pandas, scipy; print('numpy', numpy.__version__, 'pandas', pandas.__version__, 'scipy', scipy.__version__)"],
                     capture_output=True, text=True).stdout)

## Part A — check the repository and rebuild every table and figure

In [ ]:
def run(*args, env=None):
    e = dict(os.environ); e.update(env or {})
    r = subprocess.run([PY, *args], cwd=REPO, env=e, capture_output=True, text=True)
    print(r.stdout[-6000:]); print(r.stderr[-3000:])
    return r.returncode

assert run("run.py", "check") == 0, "self-check failed"   # expect: CHECK PASSED

In [ ]:
# Fingerprint every published table, result file and figure-data CSV BEFORE rebuilding (works with or without git)
import hashlib
PATTERNS = ["paper/tables*.tex", "paper/supp/*.tex",                                  # tables (run.py tables)
            "paper/results/*.csv", "paper/results/*.txt", "paper/results/*.md",         # result files (run.py tables)
            "paper/raw/aud_accounting*.csv", "paper/raw/aud_rq4_*.csv",                  # result files the table builders
            "paper/raw/aud_variants*.csv", "paper/raw/aud_S2.csv",                       #   write to paper/raw/
            "paper/figures/data/figure*.csv"]                                            # figure data (run.py figures)
def fingerprints():
    out = {}
    for pat in PATTERNS:
        for f in glob.glob(os.path.join(REPO, pat)):
            data = open(f, "rb").read().replace(b"\r\n", b"\n")
            out[os.path.relpath(f, REPO)] = hashlib.sha256(data).hexdigest()
    return out
published = fingerprints()
print(len(published), "published files fingerprinted")

In [ ]:
rc = run("run.py", "tables")      # about 7 minutes; rebuilds paper/tables_*.tex, paper/supp/*.tex, paper/results/, paper/raw/aud_*.csv
assert rc == 0
rc = run("run.py", "figures")     # about 1 minute; rebuilds paper/figures/figure1-9 and paper/figures/data/*.csv
assert rc == 0

In [ ]:
rc = run("run.py", "check")
assert rc == 0, "self-check failed after rebuilding"
rebuilt = fingerprints()
changed = sorted(f for f in published if rebuilt.get(f) != published[f])
print(len(published), "files compared;", len(changed), "differ")
for f in changed: print("  differs:", f)
assert not changed, "rebuilt outputs differ from the published ones"
print("PART A PASSED: every table, result file and figure-data file was rebuilt and is identical to the published version.")

In [ ]:
# Look at one rebuilt table: the complete-data results (manuscript Table 2)
print(open(os.path.join(REPO, "paper", "tables_rq2.tex"), encoding="utf-8").read()[:3000])

## Part B (optional) — rebuild the data from the raw files and re-run experiments
Needs `DATA_ZIP` (the XMarket ratings zip, about 330 MB, e.g. in your Google Drive). Skipped when `DATA_ZIP` is empty.

In [ ]:
HAVE_DATA = bool(DATA_ZIP) and os.path.exists(DATA_ZIP)
print("raw data found" if HAVE_DATA else ("DATA_ZIP is empty: skipping Part B" if not DATA_ZIP else f"raw data not found at {DATA_ZIP}: skipping Part B"))
if HAVE_DATA:
    os.makedirs(os.path.join(REPO, "raw_data"), exist_ok=True)
    link = os.path.join(REPO, "raw_data", "xmarket_categories.zip")
    if not os.path.exists(link):
        os.symlink(DATA_ZIP, link)
    assert run("run.py", "data", "--only", "paper", "calendar", "nvc") == 0   # about 5 minutes
    assert run("run.py", "check") == 0     # the rebuilt data summaries must equal the published ones
    assert run("-m", "pytest", "-q", "tests") == 0

In [ ]:
if HAVE_DATA:
    scratch = {"PAPER_RAW": os.path.join(REPO, "scratch", "raw"),
               "ONLY": "jp Musical_Instruments,mx Office_Products"}
    # complete-data experiment (Table 2): local, pooled, weighted and greedy arms for ItemKNN
    assert run("paper/code/rq2_run.py", "30000", "ItemKNN", "random", "full", env=scratch) == 0
    # product-launch experiment (Table 5)
    # same arguments as the paper run (python run.py list): LIMIT_SEC MODEL SPLIT MAXCOST
    assert run("paper/code/launch3.py", "1e9", "ItemKNN", "random", "1e9", env=scratch) == 0
    # compare every re-run record with the published record
    rc = run("compare_records.py", os.path.join(REPO, "scratch", "raw"), os.path.join(REPO, "paper", "raw"))
    assert rc == 0, "re-run records differ from the published ones"
    print("PART B PASSED: re-running the experiments from the raw data reproduces the published records.")

## What the results mean
* **PART A PASSED**: all tables, supplementary tables, result files and figure data in the paper are produced by
  this code from the published experiment records (the figures are redrawn from that figure data).
* **PART B PASSED**: re-running experiments from the raw XMarket files gives exactly the published records
  (more contexts or experiments: see the README, section *Re-run a few contexts*).